In [46]:
import polars as pl

In [47]:
books_df = pl.scan_ndjson("../processed-data/cleaned_books_fantasy_paranormal.json")
interactions_df = pl.scan_ndjson("../processed-data/cleaned_interactions_fantasy_paranormal.json")
authors_df = pl.scan_ndjson("../raw-data/goodreads_book_authors.json")
series_df = pl.scan_ndjson("../raw-data/goodreads_book_series.json")

# Ensure join keys have consistent dtypes (some GoodReads dumps mix int/str ids)
books_df = books_df.with_columns(
    pl.col("book_id").cast(pl.Int64, strict=False),
    pl.col("work_id").cast(pl.Int64, strict=False),
    pl.col("ratings_count").cast(pl.Int64, strict=False),
)
interactions_df = interactions_df.with_columns(
    pl.col("book_id").cast(pl.Int64, strict=False)
)

In [48]:
# filter out descriptions with fewer than 50 words
books_df = books_df.filter(pl.col('description').str.count_matches(r"\w+") >= 50)

In [49]:
books_df.select('language_code').unique().show()

language_code
str
"""por"""
"""ger"""
"""en-US"""
"""ita"""
"""rus"""


In [50]:
# Keep only english variations
# """en"""
# """en-CA"""
# """en-GB"""
# """en-US"""
# """eng"""

books_df = books_df.filter(
    pl.col('language_code').is_in(['en', 'en-CA', 'en-GB', 'en-US', 'eng'])
)


In [51]:
# Deduplicate books by work_id (keep a single representative edition per work).
# Heuristic: prefer higher ratings_count, then longer description, then lower book_id for determinism.
books_df = (
    books_df
    .with_columns(
        pl.col("description").fill_null("").str.len_chars().alias("_desc_len")
    )
    .sort(["work_id", "ratings_count", "_desc_len", "book_id"], descending=[False, True, True, False])
    .unique(subset=["work_id"], keep="first")
    .drop(["_desc_len"])
)

In [52]:
books_df = books_df.explode("authors").with_columns(
    pl.col("authors").struct.field("author_id").alias("author_id")
)
books_df = books_df.join(
    authors_df.select(["author_id", "name"]),
    on="author_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["authors", "author_id", "name"]).first(),
    pl.col("name").drop_nulls().alias("author_names")
)


In [53]:
books_df = books_df.explode("series").with_columns(
    pl.col("series").alias("series_id")
)
books_df = books_df.join(
    series_df.select(["series_id", pl.col("title").alias("series_title")]),
    on="series_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["series", "series_id", "series_title"]).first(),
    pl.col("series_title").drop_nulls().alias("series")
)


In [54]:
books_df = books_df.with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().struct.field("name")
    ).alias("popular_shelves")
).with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().filter(
            ~pl.element().str.contains(r"(?i)read|own|buy|fav|library|audio|kindle|ebook")
        )
    )
)


In [55]:
books_df.select('language_code').unique().show()

language_code
str
"""en-US"""
"""eng"""
"""en"""
"""en-GB"""
"""en-CA"""


In [56]:
books_df.head().show()

book_id,isbn,text_reviews_count,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,ratings_count,work_id,title,title_without_series,author_names,series
i64,str,i64,str,str,list[str],str,str,f64,str,list[str],str,str,str,str,i64,i64,str,i64,str,i64,str,str,i64,i64,str,str,list[str],list[str]
22042412,"""""",32,"""US""","""eng""","[""paranormal"", ""shifters"", … ""alpha-male-literal""]","""B00JWCBQAA""","""true""",3.86,"""B00JWCBQAA""","[""22930159"", ""23438958"", … ""23347641""]","""Rosalee Bennett comes to the t…","""""","""https://www.goodreads.com/book…","""""",null,null,"""""",null,"""""",null,"""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…",956,41365663,"""The Alpha's Mate (Grey Wolf Pa…","""The Alpha's Mate (Grey Wolf Pa…","[""E.A. Price""]","[""Grey Wolf Pack""]"
8316124,"""""",5,"""US""","""eng""","[""vampires"", ""paranormal"", … ""love-it""]","""""","""false""",4.12,"""""","[""771712"", ""1295060"", … ""508564""]","""O Mundo da Noite nao e um luga…","""Paperback""","""https://www.goodreads.com/book…","""Planeta""",240,null,"""9789896570866""",6,"""""",2010,"""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…",26,285262,"""A Escolhida (O Mundo da Noite,…","""A Escolhida (O Mundo da Noite,…","[""L.J. Smith""]","[""Night World""]"
20588085,"""""",29,"""US""","""eng""","[""mystery"", ""cozy-mysteries"", … ""on-my-kobo-now""]","""B00H9WPHXM""","""true""",3.72,"""B00H9WPHXM""","[""25903264"", ""25739009"", … ""25295939""]","""Darcy Sweet isn't what you wou…","""""","""https://www.goodreads.com/book…","""""",null,null,"""""",null,"""""",null,"""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…",311,39860654,"""Mists of the Past (Darcy Sweet…","""Mists of the Past (Darcy Sweet…","[""K.J. Emrick""]","[""Darcy Sweet Mystery""]"
33441,"""006056668X""",1824,"""US""","""eng""","[""fiction"", ""humor"", … ""abandoned""]","""""","""false""",3.74,"""B000OVLK04""","[""92113"", ""138054"", … ""110778""]","""Just why do humpback whales si…","""Paperback""","""https://www.goodreads.com/book…","""Harper""",321,15,"""9780060566685""",6,"""""",2004,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",33002,1684116,"""Fluke: Or, I Know Why the Wing…","""Fluke: Or, I Know Why the Wing…","[""Christopher Moore""]",[]
13611487,"""1781840881""",14,"""US""","""eng""","[""m-m"", ""shifters"", … ""what-paranormal-romance""]","""""","""true""",3.74,"""""","[""15808203"", ""18049584"", … ""15993721""]","""When wolf Pete Johnson hears t…","""ebook""","""https://www.goodreads.com/book…","""Total E-Bound""",92,1,"""9781781840887""",10,"""""",2012,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",197,19210108,"""Pete's Persuasion (Shifters' H…","""Pete's Persuasion (Shifters' H…","[""Lavinia Lewis""]","[""Shifters' Haven""]"


In [57]:
books_df = books_df.drop([
    'isbn', 'text_reviews_count', 'country_code', 'language_code',
    'is_ebook', 'kindle_asin', 'format', 'num_pages',
    'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'image_url',
    'title_without_series', 'ratings_count', 'publisher', 'similar_books', 'asin'
])
books_df.head().show()


book_id,popular_shelves,average_rating,description,link,url,work_id,title,author_names,series
i64,list[str],f64,str,str,str,i64,str,list[str],list[str]
27047564,"[""romance"", ""paranormal"", … ""free-books-for-review""]",4.38,"""**This book is part of a stand…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",47088026,"""No Place Like Home (Tallenmere…","[""Mysti Parker""]",[]
17673399,"[""paranormal"", ""romance"", … ""violence""]",3.84,"""One look will change their liv…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",24673387,"""The Loneliest Alpha (The MacKe…","[""T.A. Grey""]","[""The MacKellen Alphas""]"
17734743,"[""vampire"", ""paranormal"", … ""short-story""]",3.35,"""**Steamy 18+** Paranormal Roma…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",24811553,"""Hidden (Chalice Blood #1)""","[""E.R. Davis"", ""Emily Ryan-Davis""]","[""Chalice Blood""]"
20818259,"[""horror"", ""short-stories"", … ""omnium-gatherum""]",4.06,"""When enigmatic poet Henry Coro…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",40164163,"""Autumn in the Abyss""","[""John Claude Smith""]",[]
17734856,"[""paranormal"", ""paranormal-romance"", … ""pnr-other""]",4.02,"""Keltor is a Guardian of the li…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",24811726,"""Keltor (Guardian Archives, #1)""","[""Jennifer Sage""]","[""Guardian Archives""]"


In [58]:
books_df = books_df.with_columns(
    pl.concat_str(
        [
            pl.col("title").fill_null(""),
            pl.col("title").fill_null(""),
            pl.col("title").fill_null(""),
            pl.col("author_names").list.join(", ").fill_null(""),
            pl.col("author_names").list.join(", ").fill_null(""),
            pl.col("series").list.join(", ").fill_null(""),
            pl.col("popular_shelves").list.join(", ").fill_null(""),
            pl.col("description").fill_null("")
        ],
        separator=" "
    ).alias("combined_text")
)


In [59]:
# Basic text preprocessing on combined_text
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOPWORDS = set(stopwords.words('english'))
stopwords_regex = r"(?i)\b(" + "|".join(STOPWORDS) + r")\b"

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    if not text:
        return ""
    return " ".join([lemmatizer.lemmatize(w) for w in text.split()])

books_df = books_df.with_columns(
    pl.col("combined_text")
    .str.to_lowercase()
    .str.replace_all(r"[^\w\s]", " ") # remove punctuation
    .str.replace_all(stopwords_regex, "") # remove stopwords
    .str.replace_all(r"\s+", " ") # remove multiple spaces
    .str.strip_chars() # trim leading/trailing spaces
    .map_elements(lemmatize_text, return_dtype=pl.String)
)

books_df.head().show()


book_id,popular_shelves,average_rating,description,link,url,work_id,title,author_names,series,combined_text
i64,list[str],f64,str,str,str,i64,str,list[str],list[str],str
32066876,"[""erotica"", ""paranormal-luv"", … ""2016-books""]",3.79,"""Amelia was rebellious. She wan…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",52711742,"""The Mistress of the Vampires (…","[""Deliaria Davis"", ""Deliaria Davis""]",[],"""mistress vampire amelia chroni…"
35218444,"[""paranormal"", ""elizabeth-hunter"", … ""biased-because-i-wrote-it""]",4.34,"""Linx Maxwell's life is on the …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",56559885,"""A Ghost in the Glamour (Linx &…","[""Elizabeth Hunter""]","[""Linx & Bogie""]","""ghost glamour linx bogie 1 gho…"
25987394,"[""paranormal"", ""mystery"", … ""series-to-start""]",3.85,"""When the going gets tough... P…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",45900369,"""Witch Is When It All Began (A …","[""Adele Abbott""]","[""A Witch P.I. Mystery""]","""witch began witch p mystery 1 …"
16053758,"[""paranormal"", ""fantasy"", … ""ya-ish""]",3.74,"""Ren can see the future through…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",21838259,"""Certainty""","[""Eileen Sharp""]",[],"""certainty certainty certainty …"
11570304,"[""series"", ""to-be-released"", … ""not-published-yet""]",4.06,"""The second installment in The …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",16511062,"""Alluvion (The Ylem Trilogy, #2…","[""Tatiana Vila""]","[""The Ylem Trilogy""]","""alluvion ylem trilogy 2 alluvi…"


In [60]:
import os
os.makedirs("../processed-data", exist_ok=True)
books_df.collect().write_ndjson("../processed-data/processed_books_texts.json")
print("Dataset successfully saved!")

Dataset successfully saved!
